# Vorhersage sekündärer Proteinstruktur mit einem Transformer-Modell

Dieses Notebook führt Sie durch das Klonen des GitHub-Repositorys, das Einrichten der Kaggle-API zum Herunterladen der Daten, das Trainieren eines einfachen Transformer-Baseline-Modells in PyTorch und die Durchführung von Vorhersagen für benutzerdefinierte Proteine.

## 1. Setup und Installation
Zuerst klonen wir das Repository und installieren die benötigten Abhängigkeiten.

In [ ]:
# Repository klonen (passen Sie den Link an Ihr GitHub-Repo an)
!git clone https://github.com/okba/bio-mlops-pipeline.git
%cd bio-mlops-pipeline

In [ ]:
# Abhängigkeiten installieren
!pip install -r requirements.txt

## 2. Kaggle-API & Daten-Download
Um die Daten von Kaggle herunterzuladen, benötigen Sie einen API-Token (`kaggle.json`).

**Anleitung:**
1. Gehen Sie auf [Kaggle.com](https://www.kaggle.com/) und loggen Sie sich ein.
2. Navigieren Sie zu Ihren Kontoeinstellungen (Account -> Create New API Token).
3. Dadurch wird eine Datei namens `kaggle.json` heruntergeladen.
4. Führen Sie die folgende Zelle aus und laden Sie Ihre `kaggle.json` hoch.

In [ ]:
import os
from google.colab import files

# Kaggle credentials hochladen, falls nicht vorhanden
if not os.path.exists('/root/.kaggle/kaggle.json'):
    print("Bitte lade deine 'kaggle.json' Datei hoch:")
    uploaded = files.upload()
    for fn in uploaded.keys():
        os.makedirs('/root/.kaggle', exist_ok=True)
        with open('/root/.kaggle/kaggle.json', 'wb') as f:
            f.write(uploaded[fn])
    !chmod 600 /root/.kaggle/kaggle.json
    print("Kaggle-API erfolgreich konfiguriert!")
else:
    print("Kaggle-API bereits konfiguriert.")

In [ ]:
# Dataset von Kaggle herunterladen und entpacken
!kaggle datasets download -d aladdinpersson/protein-secondary-structure -p ./data --unzip

## 3. Datenvorbereitung
Wir importieren unsere Bereinigungs- und Aufteilungsfunktionen aus der MLOps-Pipeline. Dabei filtern wir Proteine, die länger als 512 Aminosäuren sind, und trennen die Daten in Train-, Val- und Test-Splits auf.

In [ ]:
from src.data_processing import prepare_data

# CSV-Dateien vorbereiten
train_path, val_path, test_path = prepare_data(
    csv_path="./data/protein_secondary_structure_data.csv",
    output_dir="./data/processed",
    max_len=512,
    test_size=0.1,
    val_size=0.1
)

## 4. Modelltraining
Wir starten das Training unseres Transformer-Baseline-Modells. In Google Colab können Sie das Training beschleunigen, indem Sie unter *Laufzeit -> Laufzeittyp ändern* die GPU (T4) aktivieren.

Für einen schnellen Test können Sie den Parameter `--subset_fraction 0.1` hinzufügen, um nur auf 10% der Daten zu trainieren.

In [ ]:
# Training starten
!python -m src.train \
    --train_csv ./data/processed/train.csv \
    --val_csv ./data/processed/val.csv \
    --test_csv ./data/processed/test.csv \
    --epochs 10 \
    --batch_size 64 \
    --d_model 128 \
    --nhead 4 \
    --num_layers 3 \
    --lr 1e-3 \
    --save_dir ./checkpoints \
    --subset_fraction 1.0

## 5. Visualisierung der Trainingsergebnisse
Wir zeigen die aufgezeichneten Verlust- und Genauigkeitskurven (learning curves) an.

In [ ]:
from IPython.display import Image, display

# Plot anzeigen
if os.path.exists('./checkpoints/learning_curves.png'):
    display(Image('./checkpoints/learning_curves.png'))
else:
    print("Lernkurven-Bild nicht gefunden. Wurde das Training bereits beendet?")

## 6. Struktur-Vorhersage für eigene Sequenzen (Inferenz)
Hier können Sie eine beliebige Aminosäuresequenz eingeben, um deren Sekundärstruktur (Q3: Helix, Faltblatt, Loop) vorhersagen zu lassen.

In [ ]:
import torch
from src.model import ProteinTransformer
from src.dataset import AA_ALPHABET
from src.predict import predict_sequence

# 1. Bestes Modell laden
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
checkpoint_path = "./checkpoints/best_model.pt"

if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    saved_args = checkpoint['args']
    
    # Modell initialisieren und Gewichte laden
    model = ProteinTransformer(
        vocab_size=len(AA_ALPHABET),
        d_model=saved_args.d_model,
        nhead=saved_args.nhead,
        num_layers=saved_args.num_layers,
        dim_feedforward=saved_args.dim_feedforward,
        num_classes=saved_args.num_classes
    ).to(device)
    model.load_state_dict(checkpoint['model_state_dict'])
    print("Trained model loaded successfully.")
    
    # Beispiel-Sequenz (Myoglobin-ähnliche Kette)
    sample_seq = "MVLSEGEWQLVLHVWAKVEADVAGHGQDILIRLFKSHPETLEKFDRFKHLKTEAEMKASEDLKKHGVTVLTALGAILKKKGHHEAELKPLAQSHATKHKIPIKYLEFISEAIIHVLHSRHPGNFGADAQGAMNKALELFRKDIAAKYKELGYQG"
    
    # Vorhersage ausführen
    pred_struct = predict_sequence(model, sample_seq, num_classes=3, device=device)
    
    print("\n--- Inferenz-Ergebnis ---")
    print(f"Eingabe-Sequenz:        {sample_seq}")
    print(f"Vorhergesagte Struktur: {pred_struct}")
    print("Legende: H = Alpha-Helix, E = Beta-Faltblatt, C = Coil/Loop")
    print("-------------------------")
else:
    print("Keine Modell-Checkpoint-Datei unter ./checkpoints/best_model.pt gefunden. Trainieren Sie zuerst das Modell!")